# 02 · Few-Shot Prompting
### Practical LLM Engineering — Module 02: Prompt Engineering

> **Objectives**
> - Why and how examples in the prompt guide model behaviour
> - The mathematics of in-context learning (ICL)
> - How to build a reusable `FewShotPrompt` class
> - How example selection, ordering, and formatting affect quality
> - Dynamic example selection using embedding similarity
> - Engineering tradeoffs: token budget vs accuracy vs latency

---


## 1. Overview

**Few-shot prompting** augments the prompt with a small number of worked examples (typically 1–10) before the actual input:

```
System: You are a sentiment classifier.

User:
  Review: "The battery lasts all day."   → Positive
  Review: "Screen cracked after a week." → Negative
  Review: "Decent for the price."        → Neutral

  Review: "Lightning-fast delivery, exactly as described."
  →
```

The model sees the pattern demonstrated by the examples and applies it to the new input — **without any weight updates**.

This is called **in-context learning (ICL)**.

### When to use few-shot over zero-shot

| Situation | Recommendation |
|---|---|
| Task has a non-obvious output format | Few-shot |
| Zero-shot accuracy is insufficient | Few-shot |
| Output style needs to match a template | Few-shot |
| Task is well-defined and model knows it | Zero-shot (simpler) |
| You have no labelled examples | Zero-shot |
| Token budget is very tight | Zero-shot |


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install anthropic openai transformers sentence-transformers torch numpy matplotlib -q

import os
import re
import json
import time
import random
import textwrap
import itertools
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn.functional as F

print("✅ Libraries ready")


In [ ]:
# ── Paste the LLM client abstraction from notebook 01 ─────────────────
# (In a real repo this would be: from utils.llm_client import get_llm)

@dataclass
class LLMResponse:
    text: str
    model: str
    input_tokens: int
    output_tokens: int
    latency_ms: float

    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class BaseLLMClient(ABC):
    @abstractmethod
    def complete(self, system: str, user: str,
                 max_tokens: int = 512, temperature: float = 0.0) -> LLMResponse: ...
    def __call__(self, system: str, user: str, **kwargs) -> LLMResponse:
        return self.complete(system, user, **kwargs)

class ClaudeClient(BaseLLMClient):
    def __init__(self, model="claude-sonnet-4-20250514", api_key=None):
        import anthropic
        self.client = anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
        self.model  = model
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        t0  = time.perf_counter()
        msg = self.client.messages.create(model=self.model, max_tokens=max_tokens,
            temperature=temperature, system=system,
            messages=[{"role": "user", "content": user}])
        return LLMResponse(msg.content[0].text, self.model,
            msg.usage.input_tokens, msg.usage.output_tokens,
            (time.perf_counter()-t0)*1000)

class OpenAIClient(BaseLLMClient):
    def __init__(self, model="gpt-4o-mini", api_key=None):
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key or os.environ.get("OPENAI_API_KEY"))
        self.model  = model
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        t0  = time.perf_counter()
        msg = self.client.chat.completions.create(model=self.model, max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role":"system","content":system},{"role":"user","content":user}])
        c = msg.choices[0]
        return LLMResponse(c.message.content, self.model,
            msg.usage.prompt_tokens, msg.usage.completion_tokens,
            (time.perf_counter()-t0)*1000)

class HuggingFaceClient(BaseLLMClient):
    def __init__(self, model="microsoft/Phi-3-mini-4k-instruct", device="auto"):
        from transformers import pipeline
        self.model_name = model
        self.pipe = pipeline("text-generation", model=model,
                              device_map=device, torch_dtype="auto")
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        msgs = [{"role":"system","content":system},{"role":"user","content":user}]
        t0   = time.perf_counter()
        out  = self.pipe(msgs, max_new_tokens=max_tokens,
                         temperature=max(temperature,1e-4),
                         do_sample=temperature>0, return_full_text=False)
        text = out[0]["generated_text"]
        if isinstance(text, list): text = text[-1]["content"]
        return LLMResponse(text, self.model_name, 0,
                           len(text.split()), (time.perf_counter()-t0)*1000)

def get_llm(backend="claude", **kwargs):
    return {"claude":ClaudeClient,"anthropic":ClaudeClient,
            "openai":OpenAIClient,"gpt":OpenAIClient,
            "hf":HuggingFaceClient,"local":HuggingFaceClient}[backend.lower()](**kwargs)

# ── Mock client (no API key needed) ──────────────────────────────────
class MockLLMClient(BaseLLMClient):
    """Deterministic mock — cycles through realistic responses."""
    _SENTIMENT = itertools.cycle(["Positive","Negative","Neutral","Mixed"])
    _INTENT    = itertools.cycle(["question","complaint","request","greeting"])
    _NER       = itertools.cycle(['{"entities":[{"text":"Apple","type":"ORG"},{"text":"Tim Cook","type":"PER"}]}',
                                   '{"entities":[{"text":"London","type":"LOC"},{"text":"2024","type":"DATE"}]}'])
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        time.sleep(0.03)
        lower = user.lower()
        if "sentiment" in lower or "positive" in lower:
            text = next(self._SENTIMENT)
        elif "intent" in lower:
            text = next(self._INTENT)
        elif "entit" in lower or "ner" in lower:
            text = next(self._NER)
        else:
            text = "The response demonstrates the requested pattern."
        return LLMResponse(text,"mock-llm",
                           len((system+user).split()),len(text.split()),30.0)

llm = MockLLMClient()
print("MockLLMClient active. Swap to get_llm('claude') when ready.")
resp = llm("You are helpful.", "Quick test")
print(f"Test: {resp.text!r}")


## 3. Background: In-Context Learning

### 3.1 What ICL actually does

Few-shot examples do **not** update the model's weights. Instead, they shift the conditional distribution over outputs:

$$
P_\theta(y \mid e_1, \ldots, e_k, x) \neq P_\theta(y \mid x)
$$

where $e_i = (\text{input}_i, \text{label}_i)$ are the in-context examples and $x$ is the test input.

The examples act as a **soft constraint** on the output space — they demonstrate the mapping the model should apply.

---

### 3.2 Why examples help more than instructions alone

Research (Min et al., 2022) shows that what matters most in few-shot examples is:

1. **The label space** — what output categories exist
2. **The input distribution** — what kinds of inputs to expect
3. **The input–output format** — how the answer should be structured

Surprisingly, **whether the labels are correct** matters less than you might expect — the format signal often dominates. This has important implications: even randomly labelled examples improve format compliance.

---

### 3.3 Scaling with number of shots

Performance generally improves with more examples, but with **diminishing returns**:

$$
\text{Acc}(k) \approx \text{Acc}_\infty \cdot \left(1 - e^{-\lambda k}\right)
$$

The point of diminishing returns varies by task complexity, but in practice:
- **1-shot**: establishes format
- **3–5 shot**: captures label distribution
- **8–16 shot**: near-saturated for most tasks
- **>16 shot**: rarely beneficial; high token cost


## 4. The `FewShotPrompt` Class

In [ ]:
@dataclass
class Example:
    """A single in-context example (input → output pair)."""
    input_text:  str
    output_text: str
    label:       str = ""   # optional ground-truth label for evaluation


@dataclass
class FewShotPrompt:
    """
    Builds few-shot prompts from a pool of examples.

    Features:
    - Fixed or dynamic example selection
    - Multiple formatting styles (plain, XML, markdown table, JSON)
    - Token budget tracking
    - Serialisable to (system, user) tuple for any LLM API
    """
    task_instruction: str
    examples:         list[Example]
    output_spec:      str = ""
    role:             str = "You are a precise, helpful AI assistant."
    format_style:     str = "plain"   # plain | xml | markdown | json
    separator:        str = "---"

    # ── Formatting ─────────────────────────────────────────────────
    def _format_example(self, ex: Example, idx: int) -> str:
        if self.format_style == "plain":
            return f"Input: {ex.input_text}\nOutput: {ex.output_text}"
        elif self.format_style == "xml":
            return (f"<example>\n"
                    f"  <input>{ex.input_text}</input>\n"
                    f"  <output>{ex.output_text}</output>\n"
                    f"</example>")
        elif self.format_style == "markdown":
            return f"| {ex.input_text} | {ex.output_text} |"
        elif self.format_style == "json":
            return json.dumps({"input": ex.input_text, "output": ex.output_text})
        return f"Input: {ex.input_text}\nOutput: {ex.output_text}"

    def _format_examples_block(self) -> str:
        if self.format_style == "markdown":
            header = "| Input | Output |\n|---|---|"
            rows   = "\n".join(self._format_example(e, i)
                                for i, e in enumerate(self.examples))
            return header + "\n" + rows
        parts = [self._format_example(e, i) for i, e in enumerate(self.examples)]
        return f"\n{self.separator}\n".join(parts)

    # ── Render ─────────────────────────────────────────────────────
    def render(self, test_input: str) -> tuple[str, str]:
        """Returns (system_prompt, user_prompt) for the LLM API."""
        parts = [self.task_instruction]
        if self.examples:
            parts.append("Examples:\n" + self._format_examples_block())
        parts.append(f"Now apply the same pattern to:\nInput: {test_input}\nOutput:")
        if self.output_spec:
            parts.insert(1, f"Output format: {self.output_spec}")
        return self.role, "\n\n".join(parts)

    def token_estimate(self, test_input: str = "") -> int:
        sys, usr = self.render(test_input or "sample input text")
        return len((sys + usr).split()) * 4 // 3   # ~1.33 tokens per word

    def display(self, test_input: str = "<your input here>"):
        sys, usr = self.render(test_input)
        print("── SYSTEM ──────────────────────────────────────────")
        print(sys)
        print("── USER ────────────────────────────────────────────")
        print(usr[:1200] + ("\n[... truncated ...]" if len(usr)>1200 else ""))
        print(f"── (~{self.token_estimate(test_input)} tokens) ────────────────────────")

    # ── Convenience ────────────────────────────────────────────────
    def with_examples(self, examples: list[Example]) -> "FewShotPrompt":
        """Return a copy with a different example set (immutable style)."""
        from dataclasses import replace
        return replace(self, examples=examples)

    def with_format(self, style: str) -> "FewShotPrompt":
        from dataclasses import replace
        return replace(self, format_style=style)


# ── Demo ──────────────────────────────────────────────────────────────
sentiment_examples = [
    Example("Absolutely love this product, works perfectly!", "Positive"),
    Example("Terrible quality, broke after one day.",          "Negative"),
    Example("Okay for the price, nothing special.",            "Neutral"),
    Example("Fast delivery but the item was scratched.",       "Mixed"),
]

fs_prompt = FewShotPrompt(
    task_instruction="Classify the sentiment of each customer review.",
    examples=sentiment_examples,
    output_spec="One word: Positive, Negative, Neutral, or Mixed.",
    role="You are a precise sentiment analysis assistant.",
)

fs_prompt.display("The packaging was gorgeous but the product smelled strange.")


In [ ]:
# ── Run it ────────────────────────────────────────────────────────────
test_reviews = [
    "The packaging was gorgeous but the product smelled strange.",
    "Works exactly as advertised, very happy!",
    "Instructions were unclear but the product is fine.",
]

print("Few-shot sentiment classification:\n")
for review in test_reviews:
    sys_p, usr_p = fs_prompt.render(review)
    resp = llm(system=sys_p, user=usr_p, max_tokens=10)
    print(f"  {review[:55]:<55} → {resp.text.strip()}")


## 5. How Many Shots? — The Accuracy vs Token Cost Tradeoff

In [ ]:
# ── Build a test harness for k-shot experiments ───────────────────────
SENTIMENT_POOL = [
    Example("Absolutely love this product, works perfectly!",   "Positive", "positive"),
    Example("Terrible quality, broke after one day.",            "Negative", "negative"),
    Example("Okay for the price, nothing special.",              "Neutral",  "neutral"),
    Example("Fast delivery but the item was scratched.",         "Mixed",    "mixed"),
    Example("Best purchase I've made this year — highly recommend!", "Positive", "positive"),
    Example("Do not waste your money on this rubbish.",          "Negative", "negative"),
    Example("It arrived on time and does what it says.",         "Neutral",  "neutral"),
    Example("Love the design, hate that the battery dies so fast.", "Mixed", "mixed"),
    Example("Exceeded all my expectations, stunning quality!",   "Positive", "positive"),
    Example("Completely broken out of the box.",                 "Negative", "negative"),
]

EVAL_SET = [
    Example("Fast shipping, exactly as described.",                     "", "positive"),
    Example("The colour is wrong and it started rusting in a week.",    "", "negative"),
    Example("Does the job, nothing to write home about.",               "", "neutral"),
    Example("Beautiful product but the instructions are confusing.",    "", "mixed"),
    Example("Five stars — absolutely flawless.",                        "", "positive"),
    Example("Returned immediately, completely unusable.",               "", "negative"),
]


def run_kshot_experiment(llm, pool, eval_set, k_values, n_trials=3):
    """
    For each k in k_values, randomly sample k examples from pool
    and measure accuracy on eval_set.  Repeat n_trials times and average.
    """
    results = {}
    for k in k_values:
        trial_accs, trial_tokens = [], []
        for trial in range(n_trials):
            random.seed(trial * 100 + k)
            examples = random.sample(pool, min(k, len(pool)))

            fsp = FewShotPrompt(
                task_instruction="Classify the sentiment of each customer review.",
                examples=examples,
                output_spec="One word: Positive, Negative, Neutral, or Mixed.",
                role="You are a precise sentiment analysis assistant.",
            )

            correct, tokens = 0, []
            for item in eval_set:
                sys_p, usr_p = fsp.render(item.input_text)
                resp = llm(system=sys_p, user=usr_p, max_tokens=10)
                pred = resp.text.strip().lower().rstrip(".")
                if pred == item.label:
                    correct += 1
                tokens.append(fsp.token_estimate(item.input_text))

            trial_accs.append(correct / len(eval_set))
            trial_tokens.append(np.mean(tokens))

        results[k] = {
            "acc_mean":    np.mean(trial_accs),
            "acc_std":     np.std(trial_accs),
            "tokens_mean": np.mean(trial_tokens),
        }
        print(f"  k={k:2d}  acc={results[k]['acc_mean']:.2f}±{results[k]['acc_std']:.2f}"
              f"  tokens≈{results[k]['tokens_mean']:.0f}")

    return results


print("k-shot accuracy vs token cost experiment:\n")
k_values = [0, 1, 2, 4, 6, 8]
kshot_results = run_kshot_experiment(llm, SENTIMENT_POOL, EVAL_SET, k_values, n_trials=3)


In [ ]:
# ── Plot k-shot results ───────────────────────────────────────────────
ks      = list(kshot_results.keys())
accs    = [kshot_results[k]["acc_mean"]    for k in ks]
stds    = [kshot_results[k]["acc_std"]     for k in ks]
tokens  = [kshot_results[k]["tokens_mean"] for k in ks]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Few-Shot: Accuracy and Token Cost vs Number of Examples", fontsize=12)

# Accuracy
ax1.errorbar(ks, accs, yerr=stds, fmt="o-", color="#3498db",
             lw=2, markersize=7, capsize=4, ecolor="#95a5a6")
ax1.set_xlabel("Number of examples (k)")
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0, 1.05)
ax1.set_title("Task accuracy")
ax1.grid(alpha=0.3)

# Token cost
ax2.bar(ks, tokens, color="#e67e22", edgecolor="white")
ax2.set_xlabel("Number of examples (k)")
ax2.set_ylabel("Estimated tokens per call")
ax2.set_title("Token cost per inference call")
ax2.grid(axis="y", alpha=0.3)

# Efficiency: accuracy per 100 tokens
eff = [a / (t / 100) if t > 0 else 0 for a, t in zip(accs, tokens)]
ax3.plot(ks, eff, "s-", color="#9b59b6", lw=2, markersize=7)
ax3.set_xlabel("Number of examples (k)")
ax3.set_ylabel("Accuracy per 100 tokens")
ax3.set_title("Token efficiency")
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Find sweet spot
best_k = ks[np.argmax(eff)]
print(f"Most token-efficient k: {best_k} examples")


## 6. Example Selection Strategies

Not all examples are equally useful. **Which** examples you include matters as much as how many.

### 6.1 Random selection

Simplest baseline — sample $k$ examples uniformly at random from the pool.

### 6.2 Stratified selection

Ensure each label class is represented proportionally.

### 6.3 Similarity-based (dynamic) selection

Retrieve the $k$ most **semantically similar** examples to the test input using embedding cosine similarity:

$$
\text{score}(e_i, x) = \frac{\phi(e_i) \cdot \phi(x)}{\|\phi(e_i)\| \cdot \|\phi(x)\|}
$$

where $\phi(\cdot)$ is a sentence embedding function.

This is the most powerful strategy — examples that closely match the test input's topic and phrasing give the model the most directly relevant demonstrations.


In [ ]:
# ── Example selection strategies ─────────────────────────────────────
class ExampleSelector:
    """Selects k examples from a pool for a given test input."""

    def __init__(self, pool: list[Example]):
        self.pool = pool

    def random_select(self, k: int, seed: int = 42) -> list[Example]:
        """Uniformly random sample."""
        rng = random.Random(seed)
        return rng.sample(self.pool, min(k, len(self.pool)))

    def stratified_select(self, k: int) -> list[Example]:
        """
        Select examples ensuring each label is represented.
        Distributes k slots across classes as evenly as possible.
        """
        by_label = {}
        for ex in self.pool:
            by_label.setdefault(ex.label, []).append(ex)

        n_classes = len(by_label)
        per_class = max(1, k // n_classes)
        selected  = []
        for label, exs in by_label.items():
            selected.extend(exs[:per_class])
        return selected[:k]

    def similarity_select(self, test_input: str, k: int,
                           embedder=None) -> list[Example]:
        """
        Select the k most semantically similar examples to test_input.
        Falls back to random selection if no embedder is provided.
        """
        if embedder is None:
            print("  (No embedder provided — falling back to random selection)")
            return self.random_select(k)

        # Embed test input and all pool examples
        pool_texts = [ex.input_text for ex in self.pool]
        all_texts  = pool_texts + [test_input]
        embeddings = embedder(all_texts)

        query_vec = embeddings[-1]
        pool_vecs = embeddings[:-1]

        # Cosine similarities
        sims = np.dot(pool_vecs, query_vec) / (
            np.linalg.norm(pool_vecs, axis=1) * np.linalg.norm(query_vec) + 1e-9
        )

        top_k_idx = np.argsort(sims)[::-1][:k]
        return [self.pool[i] for i in top_k_idx]


# ── Demo: compare selection strategies ───────────────────────────────
selector = ExampleSelector(SENTIMENT_POOL)
test_inp = "The product is beautifully designed but overpriced."

print(f'Test input: "{test_inp}"\n')

random_exs = selector.random_select(k=3, seed=7)
print("Random selection:")
for ex in random_exs:
    print(f"  [{ex.output_text}] {ex.input_text}")

print()
strat_exs = selector.stratified_select(k=4)
print("Stratified selection (one per class):")
for ex in strat_exs:
    print(f"  [{ex.output_text}] {ex.input_text}")


In [ ]:
# ── Similarity-based selection (requires sentence-transformers) ───────
# Uncomment when running on Colab with GPU:
# from sentence_transformers import SentenceTransformer
# st_model = SentenceTransformer("all-MiniLM-L6-v2")
# def embedder(texts):
#     embs = st_model.encode(texts, normalize_embeddings=True)
#     return embs

# ── Simulate with random unit vectors for the mock environment ────────
def mock_embedder(texts: list[str]) -> np.ndarray:
    """Deterministic pseudo-embeddings based on word hashes (demo only)."""
    np.random.seed(42)
    dim = 64
    vecs = []
    for text in texts:
        rng = np.random.RandomState(hash(text) % (2**31))
        v   = rng.randn(dim).astype(np.float32)
        vecs.append(v / np.linalg.norm(v))
    return np.array(vecs)

sim_exs = selector.similarity_select(test_inp, k=3, embedder=mock_embedder)
print("Similarity-based selection:")
for ex in sim_exs:
    print(f"  [{ex.output_text}] {ex.input_text}")

print()
print("💡 Replace mock_embedder with a real SentenceTransformer for production use.")


In [ ]:
# ── Compare accuracy across selection strategies ─────────────────────
strategies = {
    "Random":     lambda inp, k: selector.random_select(k),
    "Stratified": lambda inp, k: selector.stratified_select(k),
    "Similarity": lambda inp, k: selector.similarity_select(inp, k, mock_embedder),
}

k = 4
n_trials = 5
print(f"Accuracy comparison — k={k}, {len(EVAL_SET)} eval examples, {n_trials} trials\n")
print(f"{'Strategy':<14} {'Acc mean':>10} {'Acc std':>9} {'Tokens':>9}")
print("-" * 46)

strategy_results = {}
for strat_name, select_fn in strategies.items():
    accs_s = []
    for trial in range(n_trials):
        random.seed(trial * 7)
        correct = 0
        for item in EVAL_SET:
            exs  = select_fn(item.input_text, k)
            fsp  = FewShotPrompt(
                task_instruction="Classify the sentiment of this review.",
                examples=exs,
                output_spec="One word: Positive, Negative, Neutral, or Mixed.",
            )
            sys_p, usr_p = fsp.render(item.input_text)
            resp = llm(system=sys_p, user=usr_p, max_tokens=10)
            if resp.text.strip().lower().rstrip(".") == item.label:
                correct += 1
        accs_s.append(correct / len(EVAL_SET))

    mean, std = np.mean(accs_s), np.std(accs_s)
    tok  = FewShotPrompt(task_instruction="Classify.",
                          examples=selector.random_select(k)).token_estimate("test")
    strategy_results[strat_name] = (mean, std)
    print(f"{strat_name:<14} {mean:>10.3f} {std:>9.3f} {tok:>9}")


## 7. Example Ordering Effects

The order of examples in the prompt can affect model output — a phenomenon known as **recency bias**: models tend to weight the last few examples more heavily.

### Ordering strategies

| Strategy | Description |
|---|---|
| **Random** | No deliberate ordering |
| **Recency** | Put the most similar example last |
| **Diversity-first** | Most diverse examples first, most similar last |
| **Label-balanced** | Interleave labels evenly to avoid class bias |


In [ ]:
def order_examples(examples: list[Example], strategy: str = "random",
                   test_input: str = "", embedder=None) -> list[Example]:
    """
    Reorder examples according to the given strategy.
    'recency'  → most similar to test input placed last
    'balanced' → labels interleaved evenly
    'random'   → shuffled
    'original' → no change
    """
    exs = list(examples)

    if strategy == "original":
        return exs

    if strategy == "random":
        random.shuffle(exs)
        return exs

    if strategy == "recency" and embedder and test_input:
        # Sort by similarity ascending; most similar ends up last
        pool_vecs = embedder([e.input_text for e in exs])
        q_vec     = embedder([test_input])[0]
        sims = pool_vecs @ q_vec / (
            np.linalg.norm(pool_vecs, axis=1) * np.linalg.norm(q_vec) + 1e-9
        )
        order = np.argsort(sims)   # least similar first
        return [exs[i] for i in order]

    if strategy == "balanced":
        # Interleave by label
        by_label = {}
        for ex in exs:
            by_label.setdefault(ex.output_text, []).append(ex)
        result, queues = [], list(by_label.values())
        while any(queues):
            for q in queues:
                if q: result.append(q.pop(0))
        return result

    return exs


# ── Visualise how order changes the prompt ────────────────────────────
test_inp = "The product looks great but shipping took three weeks."
exs_4    = selector.stratified_select(k=4)

print("Same 4 examples, different orderings:\n")
for strat in ["original", "random", "balanced", "recency"]:
    ordered = order_examples(exs_4, strat, test_inp, mock_embedder)
    labels  = [e.output_text for e in ordered]
    print(f"  {strat:<10} → {labels}")


## 8. Example Format Styles

How you present examples affects both **model performance** and **token efficiency**.

Four common styles:
- **Plain** — `Input: ... / Output: ...`
- **XML** — `<example><input>...</input><output>...</output></example>`
- **Markdown table** — `| input | output |`
- **JSON** — `{"input": "...", "output": "..."}`


In [ ]:
# ── Compare format styles on the same examples ───────────────────────
exs_3 = SENTIMENT_POOL[:3]
test  = "The screen is sharp but the keyboard feels cheap."

format_styles = ["plain", "xml", "markdown", "json"]
print(f"Format comparison — {len(exs_3)} examples\n")
print(f"{'Style':<12} {'Tokens':>8}  {'Prompt excerpt (first 150 chars)'}")
print("-" * 70)

for style in format_styles:
    fsp = FewShotPrompt(
        task_instruction="Classify sentiment.",
        examples=exs_3,
        output_spec="One word: Positive, Negative, Neutral, or Mixed.",
        format_style=style,
    )
    _, usr = fsp.render(test)
    toks   = fsp.token_estimate(test)
    print(f"{style:<12} {toks:>8}  {usr[:150].replace(chr(10),' ')[:150]}")

print()
print("💡 XML/plain tend to be most reliable. Markdown is compact. JSON is machine-readable.")


In [ ]:
# ── Full rendering: XML style ─────────────────────────────────────────
fsp_xml = FewShotPrompt(
    task_instruction="Classify the sentiment of each customer review.",
    examples=SENTIMENT_POOL[:4],
    output_spec="One word: Positive, Negative, Neutral, or Mixed.",
    role="You are a precise sentiment analysis assistant.",
    format_style="xml",
)
fsp_xml.display(test)


## 9. Applying Few-Shot to Complex Output: Named Entity Recognition

Few-shot prompting shines when the output structure is non-trivial.
Here we use it for NER — extracting typed entities from text.


In [ ]:
# ── NER few-shot prompt ───────────────────────────────────────────────
NER_EXAMPLES = [
    Example(
        input_text  = "Apple CEO Tim Cook announced new products in Cupertino last Tuesday.",
        output_text = '''{"entities": [
  {"text": "Apple",     "type": "ORG"},
  {"text": "Tim Cook",  "type": "PER"},
  {"text": "Cupertino", "type": "LOC"},
  {"text": "last Tuesday", "type": "DATE"}
]}''',
    ),
    Example(
        input_text  = "The WHO declared a public health emergency in Geneva on March 11, 2020.",
        output_text = '''{"entities": [
  {"text": "WHO",     "type": "ORG"},
  {"text": "Geneva",  "type": "LOC"},
  {"text": "March 11, 2020", "type": "DATE"}
]}''',
    ),
    Example(
        input_text  = "Elon Musk's company SpaceX successfully launched Starship from Texas.",
        output_text = '''{"entities": [
  {"text": "Elon Musk", "type": "PER"},
  {"text": "SpaceX",    "type": "ORG"},
  {"text": "Starship",  "type": "PRODUCT"},
  {"text": "Texas",     "type": "LOC"}
]}''',
    ),
]

ner_prompt = FewShotPrompt(
    role="You are a precise named entity recognition (NER) assistant.",
    task_instruction="""Extract all named entities from the text.
Entity types: PER (person), ORG (organisation), LOC (location), DATE, PRODUCT, MISC.""",
    examples=NER_EXAMPLES,
    output_spec="Respond with a JSON object only. No markdown fences, no explanation.",
    format_style="plain",
)

# Test inputs
ner_tests = [
    "Microsoft and Google announced a joint AI research initiative in Seattle yesterday.",
    "Prime Minister Albanese signed the agreement at Parliament House in Canberra on Friday.",
]

print("Named Entity Recognition — few-shot\n")
for text in ner_tests:
    sys_p, usr_p = ner_prompt.render(text)
    resp = llm(system=sys_p, user=usr_p, max_tokens=200)
    print(f"Input : {text}")
    print(f"Output: {resp.text}")
    try:
        parsed = json.loads(resp.text)
        entities = parsed.get("entities", [])
        print(f"  → {len(entities)} entities: {[e['text'] for e in entities]}")
    except json.JSONDecodeError:
        print("  → JSON parse failed")
    print()


## 10. Full Dynamic Few-Shot Pipeline

In [ ]:
class DynamicFewShotPipeline:
    """
    Production-ready few-shot pipeline with:
    - Embedding-based dynamic example selection
    - Configurable k, format, ordering
    - Per-call metrics logging
    """
    def __init__(self,
                 llm:              BaseLLMClient,
                 example_pool:     list[Example],
                 task_instruction:  str,
                 output_spec:       str = "",
                 role:              str = "You are a helpful assistant.",
                 k:                 int = 4,
                 format_style:      str = "plain",
                 order_strategy:    str = "recency",
                 embedder=None,
                 max_tokens:        int = 64):

        self.llm       = llm
        self.selector  = ExampleSelector(example_pool)
        self.task      = task_instruction
        self.output    = output_spec
        self.role      = role
        self.k         = k
        self.format    = format_style
        self.order     = order_strategy
        self.embedder  = embedder
        self.max_tokens = max_tokens
        self._log      = []

    def predict(self, input_text: str) -> dict:
        """Run the pipeline for a single input. Returns prediction + metadata."""
        # 1. Select examples
        examples = self.selector.similarity_select(
            input_text, self.k, self.embedder
        ) if self.embedder else self.selector.stratified_select(self.k)

        # 2. Order examples
        examples = order_examples(examples, self.order, input_text, self.embedder)

        # 3. Build prompt
        fsp = FewShotPrompt(
            task_instruction=self.task,
            examples=examples,
            output_spec=self.output,
            role=self.role,
            format_style=self.format,
        )
        sys_p, usr_p = fsp.render(input_text)

        # 4. Call LLM
        resp = self.llm(system=sys_p, user=usr_p, max_tokens=self.max_tokens)

        result = {
            "input":          input_text,
            "prediction":     resp.text.strip(),
            "examples_used":  [e.input_text[:40] for e in examples],
            "tokens":         resp.total_tokens,
            "latency_ms":     resp.latency_ms,
        }
        self._log.append(result)
        return result

    def batch_predict(self, inputs: list[str]) -> list[dict]:
        return [self.predict(inp) for inp in inputs]

    def metrics(self) -> dict:
        if not self._log: return {}
        return {
            "n_calls":        len(self._log),
            "avg_tokens":     np.mean([r["tokens"]     for r in self._log]),
            "avg_latency_ms": np.mean([r["latency_ms"] for r in self._log]),
            "total_tokens":   sum(r["tokens"]          for r in self._log),
        }


# ── Run the pipeline ──────────────────────────────────────────────────
pipeline = DynamicFewShotPipeline(
    llm=llm,
    example_pool=SENTIMENT_POOL,
    task_instruction="Classify the sentiment of this customer review.",
    output_spec="One word: Positive, Negative, Neutral, or Mixed.",
    role="You are a precise sentiment analysis assistant.",
    k=4,
    format_style="plain",
    order_strategy="recency",
    embedder=mock_embedder,
)

test_batch = [
    "Brilliant product, zero complaints.",
    "The motor burned out after two uses.",
    "Neither great nor terrible — mediocre at best.",
    "Stunning design but way too expensive for what it is.",
]

results = pipeline.batch_predict(test_batch)
print("Dynamic Few-Shot Pipeline — batch predictions\n")
for r in results:
    print(f"  Input      : {r['input'][:55]}")
    print(f"  Prediction : {r['prediction']}")
    print(f"  Examples   : {r['examples_used'][:2]}")
    print(f"  Tokens     : {r['tokens']}  |  Latency: {r['latency_ms']:.0f}ms")
    print()

print("Pipeline metrics:", pipeline.metrics())


## 11. Engineering Insights

### 11.1 Token Budget Planning

Few-shot prompts grow linearly with $k$. For high-volume systems, this adds up quickly.

$$
\text{tokens per call} \approx T_{\text{system}} + k \cdot T_{\text{example}} + T_{\text{input}} + T_{\text{output}}
$$


In [ ]:
# ── Token budget breakdown ───────────────────────────────────────────
T_system   = 30    # system prompt
T_example  = 25    # avg tokens per example (input + output)
T_input    = 20    # avg test input
T_output   = 5     # expected output (single label)

price_per_M = 3.0  # USD per 1M input tokens (Claude Sonnet approx)
calls_per_day = 50_000

print(f"Token budget at {calls_per_day:,} calls/day")
print(f"{'k':>4} {'Tokens/call':>13} {'Daily tokens':>14} {'Daily cost ($)':>16}")
print("-" * 52)
for k in [0, 1, 2, 4, 8, 16]:
    tpc  = T_system + k * T_example + T_input + T_output
    dtok = tpc * calls_per_day
    cost = dtok * price_per_M / 1_000_000
    marker = " ← sweet spot" if k == 4 else ""
    print(f"{k:>4} {tpc:>13,} {dtok:>14,} {cost:>14.2f}{marker}")


### 11.2 Caching Few-Shot Prefixes

Many LLM APIs support **prompt caching** — if the first $N$ tokens are identical across calls (system prompt + examples), they are computed once and cached.

With **static** few-shot examples, you can often cache the entire example block:

```
[System + Task + Examples]     ← cached (billed once)
[Test Input]                   ← computed per call
```

Anthropic's Claude supports prompt caching with a `cache_control` parameter.
OpenAI's GPT-4o caches automatically for prompts > 1024 tokens.

**Implication:** static example selection + long example blocks can be cheaper than dynamic selection despite higher per-call token counts.

### 11.3 Few-Shot vs Fine-Tuning Decision

| Factor | Use Few-Shot | Use Fine-Tuning |
|---|---|---|
| Data available | < 100 examples | > 500–1000 examples |
| Task frequency | Occasional | Very high volume |
| Latency requirement | Flexible | Strict (< 100ms) |
| Prompt token budget | Affordable | Too expensive |
| Task complexity | Moderate | Very specialised |
| Model access | API only | Full model access |


## 12. Exercises

1. **Format ablation:** Run the NER prompt with all four format styles (plain, xml, markdown, json) on 5 test sentences. Score each on JSON parse success rate and entity recall. Which style is most reliable?

2. **Ordering experiment:** For the sentiment task, compare four orderings (original, random, balanced, recency) on 20 examples using real labels. Does recency bias measurably affect accuracy?

3. **Label corruption:** Deliberately flip 50% of the labels in your few-shot examples (e.g. label a positive review as Negative). Run against 10 eval examples. How much does accuracy drop? What does this tell you about what ICL actually learns?

4. **Dynamic pipeline:** Swap `mock_embedder` for a real `SentenceTransformer("all-MiniLM-L6-v2")` model. Compare accuracy of random vs similarity-based selection on 20 eval examples. Does real embedding similarity improve results?

5. **Caching economics:** Given a prompt with 6 static examples (~150 tokens), calculate the break-even number of daily API calls where prompt caching becomes cost-effective, assuming a 50% cache discount on input tokens.

---

## 13. Key Takeaways

| Concept | Summary |
|---|---|
| **Few-shot / ICL** | Examples shift output distribution without weight updates |
| **What examples teach** | Label space, input distribution, and output format — not just label correctness |
| **Optimal k** | Typically 4–8 examples; diminishing returns beyond that |
| **Selection matters** | Similarity-based > stratified > random for most tasks |
| **Ordering matters** | Recency bias is real — put most relevant example last |
| **Format style** | Plain and XML most reliable; markdown most compact |
| **Token cost** | Scales linearly with k — plan your budget |
| **Caching** | Static examples + prompt caching can offset high token counts |
| **vs fine-tuning** | Few-shot when data is scarce; fine-tune when volume is high |

---